In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install duckdb -q
import duckdb
con = duckdb.connect()

In [ ]:
#  Build raw → staging → clean (the pipeline)

con.execute("DROP TABLE IF EXISTS raw_utilization")
con.execute("DROP TABLE IF EXISTS stg_utilization")
con.execute("DROP TABLE IF EXISTS clean_utilization")

# raw: exactly as uploaded, untouched
con.execute("""
    CREATE TABLE raw_utilization AS
    SELECT * FROM 'global_pharmacy_sales_2020_2025_daily_dataset.csv'
""")

# staging: typed correctly, exact duplicate rows removed
con.execute("""
    CREATE TABLE stg_utilization AS
    SELECT DISTINCT
        CAST(date AS DATE) AS date,
        region, country, category, medicine, age_group,
        CAST(units_sold AS DOUBLE) AS units_sold,
        CAST(unit_price AS DOUBLE) AS unit_price,
        CAST(stock_level AS DOUBLE) AS stock_level,
        CAST(covid_flag AS INTEGER) AS covid_flag
    FROM raw_utilization
""")

# clean: validated — no nulls, no negative sales
con.execute("""
    CREATE TABLE clean_utilization AS
    SELECT * FROM stg_utilization
    WHERE units_sold IS NOT NULL AND units_sold >= 0 AND stock_level IS NOT NULL
""")

print("raw:", con.execute("SELECT COUNT(*) FROM raw_utilization").fetchone()[0])
print("stg:", con.execute("SELECT COUNT(*) FROM stg_utilization").fetchone()[0])
print("clean:", con.execute("SELECT COUNT(*) FROM clean_utilization").fetchone()[0])

In [ ]:
con.execute("DROP TABLE IF EXISTS fact_drug_uptake")

con.execute("""
    CREATE TABLE fact_drug_uptake AS

    WITH first_seen AS (
        SELECT
            medicine,
            region,
            MIN(date) AS launch_proxy_date
        FROM clean_utilization
        GROUP BY medicine, region
    ),

    monthly_sales AS (
        SELECT
            medicine AS drug_id,
            region,
            DATE_TRUNC('month', date) AS month,
            SUM(units_sold) AS utilization
        FROM clean_utilization
        GROUP BY
            medicine,
            region,
            DATE_TRUNC('month', date)
    )

    SELECT
        m.drug_id,
        m.region,
        m.month,
        m.utilization,
        DATEDIFF(
            'month',
            DATE_TRUNC('month', f.launch_proxy_date),
            m.month
        ) AS months_since_launch

    FROM monthly_sales m

    JOIN first_seen f
        ON m.drug_id = f.medicine
       AND m.region = f.region
""")

print(
    "fact_drug_uptake rows:",
    con.execute(
        "SELECT COUNT(*) FROM fact_drug_uptake"
    ).fetchone()[0]
)

display(
    con.execute("""
        SELECT *
        FROM fact_drug_uptake
        ORDER BY drug_id, region, month
        LIMIT 20
    """).df()
)

In [ ]:
# ============================================
# STEP 4 — Correct monthly analytical table
# ============================================

con.execute("DROP TABLE IF EXISTS fact_drug_uptake")

con.execute("""
CREATE TABLE fact_drug_uptake AS

WITH first_seen AS (
    SELECT
        medicine,
        region,
        MIN(date) AS launch_proxy_date
    FROM clean_utilization
    GROUP BY medicine, region
),

monthly_sales AS (
    SELECT
        medicine AS drug_id,
        region,
        DATE_TRUNC('month', date) AS month,
        SUM(units_sold) AS utilization
    FROM clean_utilization
    GROUP BY
        medicine,
        region,
        DATE_TRUNC('month', date)
)

SELECT
    m.drug_id,
    m.region,
    m.month,
    m.utilization,

    DATEDIFF(
        'month',
        DATE_TRUNC('month', f.launch_proxy_date),
        m.month
    ) AS months_since_launch

FROM monthly_sales m

LEFT JOIN first_seen f
    ON m.drug_id = f.medicine
    AND m.region = f.region

ORDER BY
    m.drug_id,
    m.region,
    m.month
""")

print(
    "Rows in fact_drug_uptake:",
    con.execute(
        "SELECT COUNT(*) FROM fact_drug_uptake"
    ).fetchone()[0]
)

display(
    con.execute("""
        SELECT *
        FROM fact_drug_uptake
        LIMIT 20
    """).df()
)

In [ ]:
# ============================================
# STEP 5 — Validate analytical grain
# ============================================

print("Unique medicines:",
      con.execute("""
          SELECT COUNT(DISTINCT drug_id)
          FROM fact_drug_uptake
      """).fetchone()[0])

print("Unique regions:",
      con.execute("""
          SELECT COUNT(DISTINCT region)
          FROM fact_drug_uptake
      """).fetchone()[0])

print("Unique months:",
      con.execute("""
          SELECT COUNT(DISTINCT month)
          FROM fact_drug_uptake
      """).fetchone()[0])

display(
    con.execute("""
        SELECT
            drug_id,
            region,
            MIN(month) AS first_month,
            MAX(month) AS last_month,
            COUNT(*) AS months_observed
        FROM fact_drug_uptake
        GROUP BY drug_id, region
        ORDER BY drug_id, region
        LIMIT 20
    """).df()
)

In [ ]:
# ============================================
# STEP 6 — Total demand by medicine
# ============================================

medicine_demand = con.execute("""
    SELECT
        drug_id,
        SUM(utilization) AS total_units,
        AVG(utilization) AS avg_monthly_units,
        MIN(utilization) AS min_monthly_units,
        MAX(utilization) AS max_monthly_units
    FROM fact_drug_uptake
    GROUP BY drug_id
    ORDER BY total_units DESC
""").df()

display(medicine_demand)

In [ ]:
# ============================================
# STEP 7 — Demand by region
# ============================================

region_demand = con.execute("""
    SELECT
        region,
        SUM(utilization) AS total_units,
        AVG(utilization) AS avg_monthly_units,
        COUNT(DISTINCT drug_id) AS medicines
    FROM fact_drug_uptake
    GROUP BY region
    ORDER BY total_units DESC
""").df()

display(region_demand)

In [ ]:
# ============================================
# STEP 8 — Monthly demand trend
# ============================================

monthly_demand = con.execute("""
    SELECT
        month,
        SUM(utilization) AS total_units
    FROM fact_drug_uptake
    GROUP BY month
    ORDER BY month
""").df()

display(monthly_demand.head(15))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

plt.plot(
    monthly_demand["month"],
    monthly_demand["total_units"]
)

plt.title("Monthly Total Drug Demand")
plt.xlabel("Month")
plt.ylabel("Units Sold")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# STEP 9 — Month-over-month demand
# ============================================

mom_demand = con.execute("""
    SELECT
        drug_id,
        region,
        month,
        utilization,

        LAG(utilization, 1) OVER (
            PARTITION BY drug_id, region
            ORDER BY month
        ) AS previous_month_demand

    FROM fact_drug_uptake

    ORDER BY drug_id, region, month
""").df()

display(mom_demand.head(20))
mom_demand = con.execute("""
    SELECT
        *,

        CASE
            WHEN previous_month_demand IS NULL THEN NULL
            WHEN previous_month_demand = 0 THEN NULL
            ELSE
                100.0 *
                (utilization - previous_month_demand)
                / previous_month_demand
        END AS mom_growth_pct

    FROM (
        SELECT
            drug_id,
            region,
            month,
            utilization,

            LAG(utilization, 1) OVER (
                PARTITION BY drug_id, region
                ORDER BY month
            ) AS previous_month_demand

        FROM fact_drug_uptake
    )

    ORDER BY drug_id, region, month
""").df()

display(mom_demand.head(20))

In [ ]:
# ============================================
# STEP 10 — 3-month rolling demand
# ============================================

rolling_demand = con.execute("""
    SELECT
        drug_id,
        region,
        month,
        utilization,

        AVG(utilization) OVER (
            PARTITION BY drug_id, region
            ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_3m_demand

    FROM fact_drug_uptake

    ORDER BY drug_id, region, month
""").df()

display(rolling_demand.head(20))

In [ ]:
missing_month_check = con.execute("""
    SELECT
        drug_id,
        region,
        MIN(month) AS first_month,
        MAX(month) AS last_month,
        COUNT(*) AS actual_months,
        DATEDIFF('month', MIN(month), MAX(month)) + 1 AS expected_months
    FROM fact_drug_uptake
    GROUP BY drug_id, region
    HAVING actual_months != expected_months
""").df()

print("Drug-region pairs with gaps:", len(missing_month_check))
display(missing_month_check)